## **INIT**

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger

logger = get_project_logger("sliver_Layer_public")
logger.info("Success! The logger is working.")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

## **Reading from bronze layer**

In [0]:
logger.info("Step 1/5: Reading data from Bronze layer...")
df_bronze =spark.table("transport_lakehouse.bronze.public_transport_plate_numbers_raw")
logger.info(f"Successfully read {df_bronze.count()} rows from Bronze.")

## Columns organization and new column aliases

In [0]:
logger.info("Step 2/5: Starting column reorganization and aliasing...")
df_silver = df_bronze.select(
  F.col("mispar_rechev").cast("int").alias("car_num"),
  F.col("kinuy_mishari").cast("string").alias("commercial_nm"),
  F.col("tozeret_cd").cast("int").alias("manufacturer_cd"),
  F.col("tozeret_nm").cast("string").alias("manufacturer_nm"),
  F.col("tzeva_cd").cast("int").alias("color_cd"),
  F.col("tzeva_rechev").cast("string").alias("color_nm"),
  F.col("sug_rechev_cd").cast("int").alias("vehicle_type_cd"),
  F.col("sug_rechev_nm").cast("string").alias("vehicle_type_nm"),
  F.col("sug_rechev_EU_cd").cast("string").alias("vehicle_type_EU_cd"),
  F.col("mishkal_kolel").cast("int").alias("total_weight"),
  F.col("mispar_mekomot").cast("int").alias("capacity_amount"),
  F.col("shnat_yitzur").cast("int").alias("prod_year"),
  F.col("tokef_dt").cast("date").alias("test_valid_until_dt"),
  F.col("bitul_cd").cast("string").alias("canceled_cd"),
  F.col("bitul_nm").cast("string").alias("canceled_nm"),
  F.col("bitul_dt").cast("date").alias("canceled_dt")
)

logger.info("Step 2/5: Finished column reorganization.")


## String Fixes

In [0]:
logger.info("Step 3/5: Starting string fixes...")
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 593,
        "מרצדס בנץ גרמניה"
    ).otherwise(F.col("manufacturer_nm"))
)

df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 712,
        "סאנגיונג ד.קוריאה"
    ).otherwise(F.col("manufacturer_nm"))
)

df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 114,
        "דימלרקריזלר גרמניה"
    ).otherwise(F.col("manufacturer_nm"))
)

df_silver = df_silver.withColumn(
    "vehicle_type_nm",
    F.when(
        F.col("vehicle_type_cd") == 614,
        "אוטובוס צבורי להסע מיוחדת"
    ).otherwise(F.col("vehicle_type_nm"))
)
logger.info("Step 3/5: Finished string fixes.")


## Order df by vehicle type

In [0]:
logger.info("Step 4/5: Strating ordering by vehicle type..")
df_silver_ordered = df_silver.orderBy(F.col("vehicle_type_cd").desc())

logger.info("Step 4/5: Finished Ordering by vehicle type.")

## Writing to silver layer

In [0]:
target_table = "transport_lakehouse.silver.silver_vehicles_public"
logger.info(f"Step 5/5: Writing final DataFrame to Silver table: {target_table}")
(
    df_silver_ordered.write 
        .mode("overwrite") 
        .format("delta") 
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)

)

logger.info("--- Silver_Vehicles_Publc Process Completed Successfully ---")